# AICIDENTIA - 02 Cleaning and exploratory data analysis

**Question:** How do we turn the raw AIID tables into clean, analysis-ready data, and what does the data look like?

> CRISP-DM phases: **Data Understanding** (EDA) and **Data Preparation** (cleaning, text assembly, entity normalisation, merges, validation)

The logic lives in `aicidentia.data.clean`, `.entities`, `.duplicates`, `.validate` and `.prepare`. This notebook runs the build and explains the results; numbers come from the results registry (`prep.*`) or the tables. The full write-up is `docs/data_preparation.md`.

In [ ]:
from IPython.display import Image, display

from aicidentia import registry
from aicidentia.config import PROJECT_ROOT, configure_logging, load_config, set_seeds

configure_logging()
cfg = load_config()
SEED = set_seeds(cfg["random_seed"])
FIG = PROJECT_ROOT / cfg["paths"]["figures"]
print(f"{cfg['project_name']} | seed {SEED} | snapshot {cfg['snapshot_date']}")

## 1. Build the clean tables

`build_processed()` loads the Excel export and snapshot (read-only), cleans, merges, normalises entities, validates, and writes four parquet files to `data/processed/`. It also creates the two manual review templates if they do not exist yet and **never fills in your answers**.

In [ ]:
from aicidentia.data.prepare import build_processed
from aicidentia.data.prepare_outputs import write_all

res = build_processed()
write_all(res)
t, f = res.tables, res.facts
display(t["incidents_clean"].head(3).T.head(25))

## 2. Validation

Every check must pass or the build stops: ids present and unique, dates valid, year/quarter consistent, text non-empty, entity roles and types valid, report links resolvable, aliases pointing to existing entities.

In [ ]:
import pandas as pd

checks = pd.DataFrame([c.__dict__ for c in res.checks])
display(checks.assign(passed=checks["passed"].map({True: "PASS", False: "FAIL"})))

## 3. Duplicates and merges

AIID's own merge list is applied (resolved transitively). Near-duplicates by text similarity are *candidates only*: decide each pair in `data/manual/duplicate_review.csv`.

In [ ]:
print(
    {
        k: f[k]
        for k in [
            "aiid_merge_pairs",
            "merge_map_size",
            "merged_ids_present_in_excel",
            "user_merges_applied",
        ]
    }
)
print("review status:", f["near_duplicate_review"])
display(
    t["near_duplicate_candidates"][["incident_a", "incident_b", "cosine", "title_a", "title_b"]]
)

## 4. Entities: aliases, types, privacy review

Aliases come from AIID's duplicate list (only where the canonical entity exists and there is no cycle), from names that differ only in case/accents/punctuation, and from your manual entries. Entity types are rule-based heuristics (`config/entity_types.yaml`); the data has no type field.

In [ ]:
print(
    {
        k: f[k]
        for k in [
            "entity_aliases_applied",
            "entity_aliases_skipped_aiid",
            "entities_before_alias",
            "entities_after_alias",
        ]
    }
)
display(
    t["alias_candidates"][
        ["alias", "canonical", "source", "incidents_alias", "incidents_canonical"]
    ]
)
display(t["alias_skipped"])
display(t["entity_table"]["entity_type"].value_counts().rename("entities").to_frame())
display(
    t["entity_table"]
    .groupby("entity_type")
    .head(5)
    .sort_values(["entity_type", "n_incidents"], ascending=[True, False])[
        ["entity_type", "entity", "n_incidents", "roles", "entity_type_rule"]
    ]
)

**Your action:** `data/manual/entity_review.csv` lists entities with person-like names. Fill `is_private_individual` with `yes`/`no`, then re-run this notebook (or `scripts/run_phase2_prepare.py`). Confirmed individuals are excluded from the entity tables, the network and the dashboard.

In [ ]:
print("privacy:", f["privacy"])
print("review status:", f["entity_review"], "| candidates by priority:", f["review_candidates"])

## 5. Reports

In [ ]:
rep = t["reports_clean"]
print(
    {
        k: f[k]
        for k in [
            "stub_reports_dropped",
            "reports_clean",
            "reports_unlinked_kept",
            "incidents_with_zero_clean_reports",
            "report_count_differs_from_published",
        ]
    }
)
display(
    rep[
        ["report_number", "source_domain", "date_published", "text_words", "incident_ids", "linked"]
    ].head()
)
display(rep["text_words"].describe().round(0).to_frame().T)

## 6. Exploratory figures

Top deployers include only entities typed `organisation` and the harmed chart only `population_group`; generic actors, technologies, abstract values and person-like entities are excluded by design. Organisations are **alleged** deployers; counts are incident records, not findings of fault.

In [ ]:
for name in [
    "fig_eda_1_incidents_per_year",
    "fig_eda_2_top_deployers",
    "fig_eda_3_top_harmed",
    "fig_eda_4_label_distributions",
]:
    display(Image(filename=FIG / f"{name}.png"))

## 7. Registry check

In [ ]:
for key in [
    "prep.incidents_clean",
    "prep.entities_after_alias",
    "prep.validation_checks",
    "prep.validation_failed",
]:
    print(key, "=", registry.get(key))